# Quick ipynb to convert latents_train to eval format

In [1]:
import json, random
from pathlib import Path

from testing.board import visual_to_fen, extract_visual

In [2]:
# Code to sample a set number from each and to remap to new filenames
PAR_DIR = "latents_train"
FILES = {
    "cloze_capture_eval-ntp_349.jsonl": "clozecapture_fba_eval_200.jsonl",
    "is_legal_eval-ntp_362.jsonl": "islegal_fba_eval_200.jsonl",
    "mat_adv_value_eval-ntp_375.jsonl": "matadvval_fba_eval_200.jsonl",
    "mobility_eval-ntp_328.jsonl": "mobility_fba_eval_200.jsonl",
    "under_attack_eval-ntp_395.jsonl": "underattack_fba_eval_200.jsonl",
}
MAX_SAMPLES = 200
OUT_DIR = "cleaned_fba"

In [3]:
def reformat_fba(folder: str):
    latents_dir = Path(PAR_DIR)
    out_dir = Path(OUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)

    for in_name, out_name in FILES.items():
        path = latents_dir / in_name
        lines = path.read_text(encoding='utf-8').splitlines()

        assert len(lines) >= MAX_SAMPLES, f"{path} has only {len(lines)} lines < {MAX_SAMPLES}"

        random.shuffle(lines)
        selected = lines[:MAX_SAMPLES]

        new_lines = []
        for line in selected:
            data = json.loads(line)
            chat = data['chat']
            user_text      = chat[1][1]
            assistant_text = chat[2][1]
            new_record = {
                "chat": [
                    ["system", "chess_task_sysprompt.txt"],
                    ["user",   user_text],
                    ["assistant", ""]
                ],
                "info": {
                    "board": visual_to_fen(extract_visual(user_text)),
                    "answer": {
                        "answer": assistant_text
                    }
                }
            }
            new_lines.append(json.dumps(new_record, ensure_ascii=False))

        out_path = out_dir / out_name
        out_path.write_text('\n'.join(new_lines), encoding='utf-8')

In [4]:
reformat_fba(PAR_DIR)

# Helper code to split up larger files

In [2]:
import json
import random
from pathlib import Path

# Load the file
input_file = Path("latents_train/bestmove_trainBC_5mm-ntp_1000000.jsonl")
lines = input_file.read_text(encoding='utf-8').splitlines()

# Shuffle randomly
random.shuffle(lines)

# Split into chunks of 100k
chunk_size = 100000
for i in range(10):
    start_idx = i * chunk_size
    end_idx = start_idx + chunk_size
    chunk_lines = lines[start_idx:end_idx]
    
    # Save each chunk
    output_file = Path(f"latents_train/latentsft_trainBC_bestmove_100k_p{i+1}.jsonl")
    output_file.write_text('\n'.join(chunk_lines), encoding='utf-8')
    print(f"Saved {len(chunk_lines)} lines to {output_file}")

Saved 100000 lines to latents_train\latentsft_trainBC_bestmove_100k_p1.jsonl
Saved 100000 lines to latents_train\latentsft_trainBC_bestmove_100k_p2.jsonl
Saved 100000 lines to latents_train\latentsft_trainBC_bestmove_100k_p3.jsonl
Saved 100000 lines to latents_train\latentsft_trainBC_bestmove_100k_p4.jsonl
Saved 100000 lines to latents_train\latentsft_trainBC_bestmove_100k_p5.jsonl
Saved 100000 lines to latents_train\latentsft_trainBC_bestmove_100k_p6.jsonl
Saved 100000 lines to latents_train\latentsft_trainBC_bestmove_100k_p7.jsonl
Saved 100000 lines to latents_train\latentsft_trainBC_bestmove_100k_p8.jsonl
Saved 100000 lines to latents_train\latentsft_trainBC_bestmove_100k_p9.jsonl
Saved 100000 lines to latents_train\latentsft_trainBC_bestmove_100k_p10.jsonl
